# Gaze, gratitude and revisable scene models

**Goal.** Reproduce the paper's Figure 5 before bars, distinguish unavailable
after code from authored transfer, and test active inquiry using admitted
Realizing Emptiness structural priors. This is a simulation of sampled
locations, not recorded eye tracking or a pixel-based vision model.

Run from the repository in the frozen project environment. Every code cell
is executed by the canonical pipeline. The [source dossier](../docs/yarbus-research.md)
records discrepancies, primary scholarship and the original artwork license.

In [ ]:
import sys
from thanks_for_nothing.notebooks import notebook_context
context = notebook_context()
from dataclasses import asdict
from importlib.metadata import version
from pathlib import Path
from tempfile import TemporaryDirectory
from pprint import pprint
import numpy as np
from IPython.display import Image, Markdown, display
from thanks_for_nothing.artifacts import write_record
from thanks_for_nothing.experiments import run_learning_experiment
print({name: version(name) for name in (
    'thanks-for-nothing', 'generalized-notation-notation', 'realizing-emptiness')})
print('Interpreter:', sys.executable)
root = context.root
if root is None:
    raise RuntimeError('Run this companion from the repository root or notebooks directory')

In [ ]:
from thanks_for_nothing.gaze_design import load_gaze_task
from thanks_for_nothing.source_yarbus import run_yarbus_cohort
task = load_gaze_task(context.specifications)
source = run_yarbus_cohort(task.design, root=root)
print('Positive/negative/neutral occupancy (%)')
print('Source positive profile:', source.positive_percent)
print('Source negative profile:', source.negative_percent)
assert source.published_before_matches
assert not source.after_intervention_available
assert np.allclose(source.positive_percent, (82.14, 10.98, 6.88))
assert np.allclose(source.negative_percent, (54.62, 11.80, 33.58))

## 1. Reproduction scope and the missing intervention

Original seed 42 is set once, alternating positive and negative templates
across five pairs of 1,000 states. The initial neutral state is counted.
Cached notebook bars disagree with this code and are not the oracle.
The paper's after bars are visually available but have no corresponding
learning or before/after loop in the preserved gaze notebook. We do not
tune parameters to those values. Original policies penalize conditional
entropy; that term alone is not expected information gain.

In [ ]:
from thanks_for_nothing.gaze import build_gaze_audit
learning = run_learning_experiment(specification_dir=context.specifications)
gaze = build_gaze_audit(learning, root=root, specification_dir=context.specifications)
assert gaze.all_controls_pass
print('Transfer records:', len(gaze.transfer), 'active episodes:', len(gaze.episodes),
      'exploratory contrasts:', len(gaze.contrasts))
pprint(gaze.controls)

In [ ]:
from thanks_for_nothing.gaze_art import copy_artwork
from thanks_for_nothing.gaze_figures import (
    yarbus_reconstruction, gaze_transfer_paths, gaze_inquiry_paths)
workspace = TemporaryDirectory(prefix='tfn-gaze-companion-')
destination = Path(workspace.name)
write_record(destination / 'data/gaze.json', gaze)
write_record(destination / 'data/study.json', learning.study)
copy_artwork(root, destination)
figure = yarbus_reconstruction(gaze, destination / 'figures')
display(Image(filename=str(destination / 'figures/yarbus_reconstruction.png'),
              alt=figure.alt))
print(figure.alt)

## 2. Transfer the trained channel, not the assessment data

Primary bridge: `A_scene = A_learned @ E`, using the admitted embedding
shown below. Each column specifies the declared latent-state weights for
a sampled scene location. Assessment-readout projection is retained
separately. Both use identical preferences, scene process and paired random
inputs. A path is one example; policy expectations and paired risk summarize
the complete study. Circle marks the first location and square the last;
straight segments join sampled states without cubic interpolation.

In [ ]:
from thanks_for_nothing.notebooks import gaze_design_markdown
display(Markdown(gaze_design_markdown(context.specifications)))

In [ ]:
from thanks_for_nothing.gaze_findings import gaze_main_findings
display(Markdown(gaze_main_findings(gaze)))
figure = gaze_transfer_paths(gaze, destination / 'figures')
display(Image(filename=str(destination / 'figures/gaze_transfer_paths.png'),
              alt=figure.alt))
print(figure.alt)

## 3. Check expected information gain independently

For each location enumerate outcomes: `H[p(o)] - E_q H[p(o|h)]` equals
expected posterior KL. This quantity vanishes for identical channels or
a known candidate. The active policy weights expected observation value
plus this information gain minus declared movement cost; it is a one-step
authored policy, not a full expected-free-energy planner.

In [ ]:
from thanks_for_nothing.gaze_math import region_prediction
channels = np.asarray(task.design.channels)
q = np.asarray(task.model.tensors.D)
prediction, information = region_prediction(channels, q)
oracle = []
for channel, marginal in zip(channels, prediction, strict=True):
    expected_kl = 0.0
    for outcome, probability in enumerate(marginal):
        posterior = channel[outcome] * q / probability
        expected_kl += probability * np.sum(posterior * np.log(posterior / q))
    oracle.append(expected_kl)
assert np.allclose(information, oracle, atol=1e-12)
_, known_gain = region_prediction(channels, np.array([1.0, 0.0, 0.0]))
identical = np.repeat(channels[:, :, :1], 3, axis=2)
_, null_gain = region_prediction(identical, q)
assert np.allclose(known_gain, 0, atol=1e-12)
assert np.allclose(null_gain, 0, atol=1e-12)
print('Information gain by region (nats):', information)

## 4. Transpose inquiry into the Realizing Emptiness setting

The admitted support and policy inventory are displayed above from the
selected declarations. Policies separate structural access from information-seeking
action. Stationary and reversal contexts share potential
outcome uniforms across policies. Priors reset for each episode; candidate
identity is stable within an episode. This is not cross-window learning.

Risk is the exact cross-entropy over **every declared location**, scored before each
update. Preferentially visited locations cannot make their own test set look
better. There is no pixel inference: context changes regional channels while
the licensed painting stays fixed as a display background.

In [ ]:
figure = gaze_inquiry_paths(gaze, destination / 'figures')
display(Image(filename=str(destination / 'figures/gaze_inquiry_paths.png'),
              alt=figure.alt))
print(figure.alt)
from thanks_for_nothing.gaze_summary_figures import gaze_inquiry_summary
figure = gaze_inquiry_summary(gaze, destination / 'figures')
display(Image(filename=str(destination / 'figures/gaze_inquiry_summary.png'),
              alt=figure.alt))
print(figure.alt)

## 5. Describe paths without treating fixations as independent subjects

Coverage, revisit fraction, distance and joint transition entropy describe
discrete sequences. Revisit fraction counts samples beyond the first visit
to each location. Region occupancy is divided by the uniform-location
baseline displayed above from the admitted regional assignment. Geometry is a declared
unit square, not degrees of visual angle. These are descriptive metrics,
without a human dwell-time or saccade-speed interpretation.

In [ ]:
from thanks_for_nothing.gaze_metrics import path_metrics
for row in gaze.episodes:
    if row.replicate == 0 and row.window == gaze.design.example_window:
        print(row.context, row.condition)
        pprint(asdict(path_metrics(row.locations, gaze.design)))

## 6. Inspect every exploratory interval, including adverse effects

Windows are aggregated within a replicate before resampling; transfer units
are complete paired agents within fixed luckiness strata. The declared gaze
contrasts have pointwise intervals and no confirmatory p-value family.
Exact sham and valuation-only nulls retain unavailable studentized intervals.
Open access, rather than an identified inquiry-specific benefit, accounts
for the main reversal advantage. The direct sampler comparisons below keep
that limitation visible.

In [ ]:
from thanks_for_nothing.gaze_findings import gaze_interval_findings
display(Markdown(gaze_interval_findings(gaze)))

## 7. Inspect a single joint family without losing the paired units

All declared contrasts enter one explicitly exploratory standardized bootstrap maximum.
Transfer columns share whole-agent indices within each fixed world. Active
windows are averaged within each replicate; context blocks draw independently.
The denominator is each ORIGINAL standard error, not a resampled bootstrap-t
standard error. Exact valuation/sham identities are checked before receiving
zero bounds. The [joint method](../docs/joint-uncertainty.md) records assumptions,
conditional quantile Monte Carlo error and complete availability semantics.


In [ ]:
from thanks_for_nothing.gaze_joint import build_gaze_uncertainty
from thanks_for_nothing.gaze_findings import (
    gaze_joint_findings, joint_calibration_findings)
joint = build_gaze_uncertainty(gaze)
assert joint.all_controls_pass
print('Independent blocks (name, fixed strata, units/stratum):', joint.region.blocks)
display(Markdown(gaze_joint_findings(joint)))
display(Markdown(joint_calibration_findings(joint)))


In [ ]:
from thanks_for_nothing.joint_figures import joint_gaze_summary, joint_coverage
write_record(destination / 'data/gaze_uncertainty.json', joint)
for draw in (
    joint_gaze_summary(gaze, joint, destination / 'figures'),
    joint_coverage(joint, destination / 'figures'),
):
    display(Image(filename=str(destination / 'figures' / (draw.name + '.png')), alt=draw.alt))
    print(draw.alt)


## 8. A sample constant is not an exact population identity

An all-zero observed vector alone cannot justify a zero-width stochastic interval.
The mathematical example below deliberately lacks a mechanism-exact license.
It stays unavailable; changing that label to obtain a pleasing interval would
change the scientific claim. Coverage checks count unavailable families as
noncoverage, and retain the centered-exponential and rare-event failures.


In [ ]:
from thanks_for_nothing.joint_contracts import JointBlock
from thanks_for_nothing.simultaneous import joint_region
accidental = JointBlock(
    'illustrative sample only', ('unknown_population_mean',),
    np.zeros((1, 4, 1), dtype=np.float64), (False,))
unknown = joint_region((accidental,), gaze.design.joint)
assert unknown.status == 'unavailable'
assert unknown.intervals[0].low is None
pprint(asdict(unknown))
workspace.cleanup()


## Takeaways and disciplined extensions

We reproduce source before occupancy, add an explicit parameter-transfer
mechanism, and model revisable structural access. More positive-looking
probability is not sufficient evidence of better prediction. A correct
restriction can remain useful. Neither an open prior nor information-seeking
software measures the realization of emptiness.

Primary background: Yarbus (1967), complex-objects chapter; Castelhano,
Mack and Henderson (2009), DOI 10.1167/9.3.6; Mirza et al. (2019), DOI
10.1038/s41598-019-50138-8. See the dossier for actual reading scope.
Artwork: Munch, *The Scream* (1893), NG.M.00939; Nasjonalmuseet / Børre
Høstland photo CC BY 4.0. Display is rescaled with model overlays.

Further hypotheses need new declared designs: empirical eye tracking,
participant-level parameter recovery, pixel-based scene models and learning
across context changes are not implemented by this notebook.